In [ ]:
import FedFisher

In [ ]:

!unzip -q /content/drive/MyDrive/PlantVillage.zip

In [ ]:
import torch

In [ ]:
import torch

RUN THIS CELL FOR SETTING UP THE DATA AND PACKAGES

In [ ]:
print("hello vip_1")
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch
# import flwr as fl
import time
import numpy as np
from torch import nn, optim
import torch.nn.functional as F
from torchvision import datasets, transforms, models
import torchvision
from collections import OrderedDict
from torch.autograd import Variable
from PIL import Image
from torch.optim import lr_scheduler
import copy
import json
import os
import shutil
from os.path import exists
import random
from numpy import random
from multiprocessing import Process, freeze_support
from tqdm import tqdm
import glob
import shutil
from torch.utils.data import Subset
from torch.utils.data import Sampler
import sys

from torch.utils.data import SubsetRandomSampler

# print(tqdm.__version__)

train_on_gpu = torch.cuda.is_available()

if not train_on_gpu:
    print('CUDA is not available.  Training on CPU ...')
else:
    print('CUDA is available!  Training on GPU ...')
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

print("hello");

# check if CUDA is available





 #for apple dataset
 #startcomment: to avoid repeated creation of dataset
""""
original_dataset_dir = '/home/mihir/Downloads/PlantVillage'       # Change this to the actual path of your dataset
custom_dataset_dir = '/home/mihir/Downloads/CustomDataset'        # Change this to the desired destination for the custom dataset

os.makedirs(os.path.join(custom_dataset_dir, 'train', 'healthy'), exist_ok=True)
os.makedirs(os.path.join(custom_dataset_dir, 'train', 'diseased'), exist_ok=True)
os.makedirs(os.path.join(custom_dataset_dir, 'test', 'healthy'), exist_ok=True)
os.makedirs(os.path.join(custom_dataset_dir, 'test', 'diseased'), exist_ok=True)

#manage healthy
healthy_source_dir_train = os.path.join(original_dataset_dir, 'train', 'Apple___healthy')
healthy_source_dir_test =  os.path.join(original_dataset_dir, 'test', 'Apple___healthy')

healthy_dest_dir_train = os.path.join(custom_dataset_dir, 'train', 'healthy')
healthy_dest_dir_test = os.path.join(custom_dataset_dir, 'test', 'healthy')


# coppy 'pv/train/apple_healthy' to 'custom/train/healthy'
healthy_images = os.listdir(healthy_source_dir_train)


for image in healthy_images:
     src = os.path.join(healthy_source_dir_train, image)
     dst = os.path.join(healthy_dest_dir_train, image)
     shutil.copy(src, dst)

# coppy 'pv/test/apple_healthy' to 'custom/test/healthy'
healthy_images_test = os.listdir(healthy_source_dir_test)


for image in healthy_images_test:
     src = os.path.join(healthy_source_dir_test, image)
     dst = os.path.join(healthy_dest_dir_test, image)
     shutil.copy(src, dst)

#manage unhealthy

#copy pv/train/**diseased** to custom/train/diseased

for root, _, files in os.walk(os.path.join(original_dataset_dir,'train')):
    if 'Apple_' in root and root != healthy_source_dir_train and root != healthy_source_dir_test:
        for file in files:
            src = os.path.join(root, file)
            dst_train = os.path.join(custom_dataset_dir, 'train', 'diseased', file)

            shutil.copy(src, dst_train)
# val

#copy pv/test/**diseased** to custom/test/diseased

for root, _, files in os.walk(os.path.join(original_dataset_dir,'test')):
    if 'Apple_' in root and root != healthy_source_dir_train and root != healthy_source_dir_test:
        for file in files:
            src = os.path.join(root, file)
            dst_test = os.path.join(custom_dataset_dir, 'test', 'diseased', file)

            shutil.copy(src, dst_test)


#endcomment: for avoidimg creating datasets repeatedly

"""

RUN THIS CELL


In [ ]:


def set_parameter_requires_grad(model, feature_extracting):
    if feature_extracting:
        for param in model.parameters():
            param.requires_grad = False

# Number of classes in the dataset
num_classes = 2

# Number of epochs to train for
num_epochs = 10

# Flag for feature extracting. When False, we finetune the whole model,
#   when True we only update the reshaped layer params
feature_extract = True


def final_classifier(num_ftrs,num_classes):
  return nn.Sequential(OrderedDict([
                          ('fc1', nn.Linear(num_ftrs, 512)),
                          ('relu', nn.ReLU()),
                          ('fc2', nn.Linear(512, num_classes)),
                          ('output', nn.Softmax(dim=1))   #changed LogSoftmax to softmax
                          ]))

def initialize_model(model_name, num_classes, feature_extract, use_pretrained=True):
    # Initialize these variables which will be set in this if statement. Each of these
    #   variables is model specific.
    model_ft = None
    input_size = 0

    if model_name == "resnet152":
        """ Resnet152
        """
        model_ft = models.resnet152(pretrained=use_pretrained)
        set_parameter_requires_grad(model_ft, feature_extract)
        num_ftrs = model_ft.fc.in_features
        print(num_ftrs)
        model_ft.fc = final_classifier(num_ftrs,num_classes)
        input_size = 224

    elif model_name == "resnet50":
        """ Resnet18
        """
        model_ft = models.resnet50(pretrained=use_pretrained)
        set_parameter_requires_grad(model_ft, feature_extract)
        num_ftrs = model_ft.fc.in_features
        print(num_ftrs)
        model_ft.fc = final_classifier(num_ftrs,num_classes)
        input_size = 224

    elif model_name == "resnet18":
        """ Resnet18
        """
        model_ft = models.resnet18(pretrained=use_pretrained)
        set_parameter_requires_grad(model_ft, feature_extract)
        num_ftrs = model_ft.fc.in_features
        print(num_ftrs)
        model_ft.fc = final_classifier(num_ftrs,num_classes)
        input_size = 224

    elif model_name == "alexnet":
        """ Alexnet
        """
        model_ft = models.alexnet(pretrained=use_pretrained)
        set_parameter_requires_grad(model_ft, feature_extract)
        num_ftrs = model_ft.classifier[6].in_features
        print(num_ftrs)
        model_ft.classifier[6] = final_classifier(num_ftrs,num_classes)
        input_size = 224

    elif model_name == "vgg":
        """ VGG11_bn
        """
        model_ft = models.vgg11_bn(pretrained=use_pretrained)
        set_parameter_requires_grad(model_ft, feature_extract)
        num_ftrs = model_ft.classifier[6].in_features
        print(num_ftrs)
        model_ft.classifier[6] = final_classifier(num_ftrs,num_classes)
        input_size = 224

    elif model_name == "squeezenet":
        """ Squeezenet
        """
        model_ft = models.squeezenet1_0(pretrained=use_pretrained)
        set_parameter_requires_grad(model_ft, feature_extract)
        num_ftrs = 512
        print(num_ftrs)
        model_ft.classifier[1] = final_classifier(num_ftrs,num_classes)
        model_ft.num_classes = num_classes
        input_size = 224

    elif model_name == "densenet":
        """ Densenet
        """
        model_ft = models.densenet121(pretrained=use_pretrained)
        set_parameter_requires_grad(model_ft, feature_extract)
        num_ftrs = model_ft.classifier.in_features
        print(num_ftrs)
        model_ft.classifier = final_classifier(num_ftrs,num_classes)
        input_size = 224

    elif model_name == "inception":
        """ Inception v3
        Be careful, expects (299,299) sized images and has auxiliary output
        """
        model_ft = models.inception_v3(pretrained=use_pretrained)
        set_parameter_requires_grad(model_ft, feature_extract)
        # Handle the auxilary net
        num_ftrs = model_ft.AuxLogits.fc.in_features
        print(num_ftrs)
        model_ft.AuxLogits.fc = final_classifier(num_ftrs,num_classes)
        # Handle the primary net
        num_ftrs = model_ft.fc.in_features
        print(num_ftrs)
        model_ft.fc = final_classifier(num_ftrs,num_classes)
        input_size = 299

    elif model_name == 'mobilenet':
        model_ft = models.mobilenet_v2(pretrained=use_pretrained)
        set_parameter_requires_grad(model_ft, feature_extract)
        num_ftrs = model_ft.classifier[1].in_features
        print(num_ftrs)
        model_ft.classifier[1] = final_classifier(num_ftrs,num_classes)
        input_size = 224

    else:
        print("Invalid model name, exiting...")
        exit()

    return model_ft, input_size


def load_checkpoint(filepath,model_to_load):
    checkpoint = torch.load(filepath,map_location='cpu')
    model,initial_size = initialize_model(model_to_load,num_classes,feature_extract,use_pretrained=True)

    model.load_state_dict(checkpoint)
    model.to(device)

    return model, checkpoint


# Load model and get index to class mapping
model_name = "densenet"
model, class_to_idx = load_checkpoint(r'/home/vip-lab/Downloads/workspace/onion_binary_densenet_pv.pt',model_name)
idx_to_class = { v : k for k,v in class_to_idx.items()}

model = model.to(device=device)

print("line 235")

print("\n")



# This line is necessary for Windows


#Organizing the dataset
# data_dir = r'/home/vip-lab/Downloads/workspace/CustomApple'
# data_dir = r'/home/vip-lab/Downloads/workspace/CustomSoya/ds'
data_dir = r'/home/vip-lab/Downloads/workspace/CustomApple'
data_dir= r'/home/vip-lab/Downloads/workspace/tih_onion'
train_dir = data_dir + '/train'
valid_dir = data_dir + '/val'
nThreads = 1
batch_size = 16
use_gpu = torch.cuda.is_available()

# Define your transforms for the training and validation sets
# Data augmentation and normalization for training
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomRotation(30),
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

transform_test = transforms.Compose([
    transforms.Normalize(mean=[0., 0., 0.], std=[1 / 0.229, 1 / 0.224, 1 / 0.225]),
    transforms.Normalize(mean=[-0.485, -0.456, -0.406], std=[1., 1., 1.])
])



def cutmix_criterion(preds, targets):
    targets1, targets2, lam = targets[0], targets[1], targets[2]
    criterion = nn.NLLLoss()
    return lam * criterion(preds, targets1) + (1 - lam) * criterion(preds, targets2)

def cutmix(data, targets, alpha):
    indices = torch.randperm(data.size(0))
    shuffled_data = data[indices]
    shuffled_targets = targets[indices]

    lam = np.random.beta(alpha, alpha)
    bbx1, bby1, bbx2, bby2 = rand_bbox(data.size(), lam)
    data[:, :, bbx1:bbx2, bby1:bby2] = data[indices, :, bbx1:bbx2, bby1:bby2]
    # adjust lambda to exactly match pixel ratio
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (data.size()[-1] * data.size()[-2]))

    new_targets = [targets, shuffled_targets, lam]
    return data, new_targets


def rand_bbox(size, lam):
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    # uniform
    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2



print("line 327")
# from torchsampler import ImbalancedDatasetSampler
from torchsampler import ImbalancedDatasetSampler
#Function to train the model
def train_model(model, criterion, optimizer, scheduler, num_epochs=20,is_inception = False):
    since = time.time()

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(1, num_epochs+1):
        print('Epoch {}/{}'.format(epoch, num_epochs))
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data.
            # for inputs, labels in tqdm(dataloaders[phase]):
            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)

                # zero the parameter gradients
                optimizer.zero_grad()
                # if (phase == 'train'):
                #     p = np.random.rand()
                #     if p < p_cutmix:
                #         inputs, labels = cutmix(inputs, labels, 0.8)


                # forward
                # track history if only in train
                with torch.set_grad_enabled(phase == 'train'):

                    if (is_inception and phase == 'train'):
                        outputs, aux_outputs = model(inputs)
                        loss1 = criterion(outputs, labels)
                        loss2 = criterion(aux_outputs, labels)
                        loss = loss1 + 0.4*loss2
                    else :
                        outputs = model(inputs)

                    _, preds = torch.max(outputs, 1)

                    # backward + optimize only if in training phase
                    if phase == 'train':
                        loss = criterion(outputs, labels)
                        # if p < p_cutmix:
                        #     loss = cutmix_criterion(outputs, labels)

                        loss.backward()
                        optimizer.step()

                    elif phase == 'val':
                        loss = criterion(outputs,labels)

                # statistics
                running_loss += loss.item() * inputs.size(0)
                if (phase == 'train'):
                    running_corrects += torch.sum(preds == labels.data)
                    # if (p < p_cutmix):
                    #     running_corrects += torch.sum(preds == labels[0].data)

                elif (phase == 'val'):
                    running_corrects += torch.sum(preds == labels.data)

            epoch_loss = ru
            # deep copy the model
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        scheduler.step()

    time_elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(
        time_elapsed // 60, time_elapsed % 60))
    print('Best valid accuracy: {:4f}'.format(best_acc))

    # load best model weights
    model.load_state_dict(best_model_wts)
    return model

# Train a model with a pre-trained network
num_epochs = 10
if use_gpu:
    print ("Using GPU: "+ str(use_gpu))
    model = model.cuda()

# NLLLoss because our output is LogSoftmax
criterion = nn.NLLLoss()

params_to_update = model.parameters()
print("Params to learn:")
if feature_extract:
    params_to_update = []
    for name,param in model.named_parameters():
        if param.requires_grad == True:
            params_to_update.append(param)
            print("\t",name)
else:
    for name,param in model.named_parameters():
        if param.requires_grad == True:
            print("\t",name)

# Adam optimizer with a learning rate
optimizer = optim.Adam(params_to_update, lr=0.001)
# Decay LR by a factor of 0.1 every 5 epochs
exp_lr_scheduler = lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
print("line 486")

"""
torch.save(model.state_dict(), 'appleDenseNetFull.pth')

print("model saved \n")

while(True):
    continue
"""
np.random.seed(2)
torch.manual_seed(2)

image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x]) for x in ['train','val']}

# Using the image datasets and the trainforms, define the dataloaders
train_len = len(image_datasets['train'])

random_mask = [random.choice([0, 1]) for _ in range(train_len)]

random_mask=torch.tensor(random_mask)
random_mask=random_mask.nonzero().reshape(-1)

#image_datasets['train']=Subset(image_datasets['train'], random_mask)


subset_size = 16000

# Create a subset of the CIFAR10 dataset

#torch.manual_seed(0)

subset_indices = torch.randperm(len(image_datasets['train']))[:subset_size]
subset_indices=list(range(1, len(image_datasets['train']), 2))
subset_indices=list(range(1, len(image_datasets['train'])))

# val


class EvenIndexSampler(Sampler):
    def __init__(self, data_source):
        self.data_source = data_source
        # if(sys.argv[2]=="even"):
        if(True):
            self.indices = list(range(0, len(data_source),2))
        else:
            self.indices = list(range(1, len(data_source),2))

    def __iter__(self):
        return iter(self.indices)

    def __len__(self):
        return len(self.indices)


even_index_sampler = EvenIndexSampler(image_datasets['train'])



train_subset_1 = Subset(image_datasets['train'], subset_indices)

#trainloader_subset_1=torch.utils.data.DataLoader(train_subset_1, sampler=ImbalancedDatasetSampler(train_subset_1),batch_size=64)
trainloader_subset_1=torch.utils.data.DataLoader(image_datasets['train'], sampler=even_index_sampler,batch_size=64,num_workers=0)

# One can use the Imbalaniced Dataset Smampler to sampleunevenly distributed classes equally during training
dataloaders = {
    x: torch.utils.data.DataLoader(image_datasets[x], sampler=ImbalancedDatasetSampler(image_datasets[x]),
                                   batch_size=1, num_workers=0) for x in ['train', 'val']}
# dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x],batch_size=batch_size, num_workers=8)for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}

class_names = image_datasets['train'].classes


dataset_size = len(image_datasets['train'])
dataset_indices = list(range(dataset_size))

np.random.shuffle(dataset_indices)


split_index = int(np.floor(0.5 * dataset_size))

train_idx, val_idx = dataset_indices[:split_index], dataset_indices[split_index:]

sampler1 = SubsetRandomSampler(train_idx)
sampler2 = SubsetRandomSampler(val_idx)

if(True):
    trainloader_subset_random=  torch.utils.data.DataLoader(dataset=image_datasets['train'], shuffle=False, batch_size=16, sampler=sampler1)
else:
    trainloader_subset_random=  torch.utils.data.DataLoader(dataset=image_datasets['train'], shuffle=False, batch_size=32, sampler=sampler2)







class_distribution = {}

for _, labels in trainloader_subset_1:
    for label in labels.numpy():  # Convert labels to a numpy array for easier handling
        if label in class_distribution:
            class_distribution[label] += 1
        else:
            class_distribution[label] = 1

print("Class Distribution:", class_distribution)







# class FlowerClient(fl.client.NumPyClient):
#     def get_parameters(self, config):
#         return [val.cpu().numpy() for _, val in net.state_dict().items()]

#     def set_parameters(self, parameters):
#         params_dict = zip(net.state_dict().keys(), parameters)
#         state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
#         net.load_state_dict(state_dict, strict=True)

#     def fit(self, parameters, config):
#         self.set_parameters(parameters)
#         train_mod(net,epochs=10)
#         return self.get_parameters(config={}), len(trainloader.dataset), {}

#     def evaluate(self, parameters, config):
#         self.set_parameters(parameters)
#         loss, accuracy = test(net, testloader)
#         return loss, len(testloader.dataset), {"accuracy": accuracy}





def test(net, testloader):
    """Validate the model on the test set."""
    criterion = torch.nn.CrossEntropyLoss()
    correct, loss = 0, 0.0
    with torch.no_grad():
        for images, labels in tqdm(testloader):
            outputs = net(images.to(device))
            labels = labels.to(device)
            loss += criterion(outputs, labels).item()
            correct += (torch.max(outputs.data, 1)[1] == labels).sum().item()
    accuracy = correct / len(testloader.dataset)
    return loss, accuracy

if (__name__ == '__main__'):

    for inputs, labels in trainloader_subset_1:
        inputs, labels = inputs.to(device), labels.to(device)
        inputs = transform_test(inputs)
        print(labels)
        inputs, labels = cutmix(inputs, labels, 0.8)

        plt.figure(figsize=(10, 10))
        plt.imshow(inputs[0].permute(1, 2, 0).cpu().numpy())
        print(labels)
        break

    model.classifier[1].fc2 = nn.Linear(512, 2)

    # print("line 350")
    # train_path = '/home/vip-lab/Downloads/workspace/CustomApple/train'
    # val_path = '/home/vip-lab/Downloads/workspace/CustomApple/val'

    # len_disease = len(os.listdir(train_path + "/diseased")) + len(os.listdir(val_path + "/diseased"))
    # len_healthy = len(os.listdir(train_path + "/healthy")) + len(os.listdir(val_path + "/healthy"))

    # print("Number of training healthy images = ", len(os.listdir(train_path + "/healthy")))
    # print("Number of training diseased images = ", len(os.listdir(train_path + "/diseased")))
    # print("Number of validation healthy images = ", len(os.listdir(val_path + "/healthy")))
    # print("Number of validation diseased images = ", len(os.listdir(val_path + "/diseased")))
    # plt.bar(['diseased', 'healthy'], [len_disease, len_healthy])
    # plt.show()

    net = model
    trainloader = dataloaders['train']
    testloader = dataloaders['val']

    p_cutmix = 0.5




    # if (True):


    #   train_mod(net, 30)
    #   loss,accuracy=test(net,testloader)
    #   print("loss="+str(loss)+"/n")
    #   print("accuracy="+str(accuracy)+"/n")



    # else:
    # #   fl.client.start_numpy_client(
    # # server_address="127.0.0.1:8080",
    # # client=FlowerClient(),
    # # )


RUN THIS FOR DEFINING FUNCTIONS

In [ ]:
def train_mod_labled(net, epochs, tl):
    """Train the model on the training set."""
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(net.parameters(), lr=0.001, momentum=0.9)
    for _ in range(epochs):
      for images, labels in tqdm(tl):
      #for images, labels in tqdm(dataloaders['train']):
        # for images, labels in dataloaders['train']:
            optimizer.zero_grad()
            criterion(net(images.to(device)), labels.to(device)).backward()
            optimizer.step()

def test_logit_output(net, testloader):
    """Validate the model on the test set."""
    outputs_list=[]
    criterion = torch.nn.CrossEntropyLoss()
    correct, loss = 0, 0.0
    with torch.no_grad():
        for images, labels in tqdm(testloader):
            outputs = net(images.to(device))
            labels = labels.to(device)
            loss += criterion(outputs, labels).item()
            correct += (torch.max(outputs.data, 1)[1] == labels).sum().item()
            outputs_list.append(outputs.cpu().numpy())
    accuracy = correct / len(testloader.dataset)
    return loss, accuracy,outputs_list
trainloader_subset_random_1=  torch.utils.data.DataLoader(dataset=image_datasets['train'], shuffle=False, batch_size=32, sampler=sampler1)

trainloader_subset_random_2=  torch.utils.data.DataLoader(dataset=image_datasets['train'], shuffle=False, batch_size=32, sampler=sampler2)


RUN THIS CELL, IT CREATES TWO SUBSETS OF THE DATA. IGNORE THE NAMES 'LABLED' and 'UNLABLES' JUST TREAT THEM AS TWO DIFFERENT SUBSETS

In [ ]:

print("hello")
# unlabled_subset=[img for i, img in enumerate(image_datasets['train'])if i%2!=0 ]
# labled_subset=  [img for i, img in enumerate(image_datasets['train'])if i%2==0 ]
# unlabled_subset_lables=  [img[1] for img in unlabled_subset]
# labled_subset_lables  =  [img[1] for img in   labled_subset]

unlabled_subset=[img for i, img in enumerate(image_datasets['train'])if i%2==0 ]
labled_subset=  [img for i, img in enumerate(image_datasets['train'])if i%2!=0]
unlabled_subset_lables=  [img[1] for img in unlabled_subset]
labled_subset_lables  =  [img[1] for img in   labled_subset]



FEDFISHER

RUN THIS CELL FOR IMPORTING RELATED PACKAGES IF YOU WANT TO DO FEDERATED LEARNING ON JETSON NANO

In [ ]:
from torch.utils.data import DataLoader, Dataset
from nngeometry.metrics import FIM
from nngeometry.object import PMatKFAC, PMatDiag, PVector
from torch.nn.utils import parameters_to_vector, vector_to_parameters
n_class=3
hidden_layer=16

RUN THIS

In [ ]:
def train_mod(net_m, epochs,loader):
    # """Train the model on the training set."""
    # net.train()
    # criterion = torch.nn.CrossEntropyLoss()
    # optimizer = torch.optim.SGD(net.parameters(), lr=0.001, momentum=0.9)
    # for _ in range(epochs):
    #   for images, labels in tqdm(trainloader_subset_1):
    #   #for images, labels in tqdm(dataloaders['train']):
    #     # for images, labels in dataloaders['train']:
    #         optimizer.zero_grad()
    #         criterion(net(images.to(device)), labels.to(device)).backward()
    #         optimizer.step()

        net_m.train()
        optimizer = torch.optim.SGD(net_m.parameters(), lr=0.001, momentum = 0.9)
        optimizer.zero_grad()
        step_count = 0
        loss_func = nn.CrossEntropyLoss()
        
        net_m.to(device)
        dataset_local=[]
        for epoch in range(epochs):
            batch_loss = []
            for batch_idx, (images, labels) in enumerate(loader):
                images, labels = images.to(device), labels.to(device)
                # if(self.args['augmentation']==True):
                #     images = self.transform_train(images)
                optimizer.zero_grad()
                
                log_probs = net_m(images)
                loss = loss_func(log_probs, labels)
                loss.backward()
                optimizer.step()
                batch_loss.append(loss.item())
                # dataset_local.append(tuple([torch.tensor(feats).squeeze(),labels]))

                # print(f"length of local dataset is {len(dataset_local)}")
                # print(f"shape of local dataset[0][0] is {dataset_local[0][0].shape}")
                # print(f"dataset_local[0][0] is {dataset_local[0][0]}")

            print ("Epoch No. ", epoch, "Loss " , sum(batch_loss)/len(batch_loss))





RUN THIS CELL TO MODIFY THE BACKBONE MODEL TO REMOVE CLASSIFIER LAYER. SO THAT IT WILL OUTPUT THE FEATURES BEFORE CLASSIFIER LAAYER. THIS BACKBONE MODEL WILL DEPEND ON THE PLANT THAT YOU ARE TARGETTING. (ONION,TOMATO, ETC). LOAD THE CORROSPONDING MODEL WITH TORCH.LOAD()

In [1]:
#Backbone model

model_densenet_pv=torch.load('/home/vip-lab/Downloads/workspace/DenseNet_Onion_model.pt',map_location=device)
# model_densenet_pv=torch.load('/home/vip-lab/Downloads/workspace/tih_onion_subset.pt',map_location=device)

densenet_pv_pt_feat=nn.Sequential(*list(model_densenet_pv.children())[:-1])

class ModifiedModel(nn.Module):
    def __init__(self, pretrained_model):
        super(ModifiedModel, self).__init__()
        self.pretrained_model = pretrained_model
        self.avg_pool = nn.AvgPool2d(kernel_size=7) 

    def forward(self, x):
      
        output = self.pretrained_model(x)
    
        output = self.avg_pool(output)
       
        output = output.view(output.size(0), -1)
        return output


pretrained_model = densenet_pv_pt_feat
    

modified_model = ModifiedModel(pretrained_model)
modified_model.to(device)

torch.manual_seed(27)
output = modified_model(torch.randn(1,3,224,224).to(device))
output.shape, output


NameError: name 'torch' is not defined

In [ ]:
#convert to onnx
# torch.onnx.export(modified_model,torch.randn(1, 3, 224, 224).to(device),"model123.onnx",export_params=True,opset_version=13) 


RUN THIS CELL TO CREATE OUR OWN CLASSIFIER, WHICH WILL TAKE INPUTS FRO THE MODEL THAT WE HAD MODIFIED ABOVE.

In [ ]:
# simple neural network

import torch
import torch.nn as nn
import torch.nn.functional as F


class Net(nn.Module):

    def __init__(self):
        super(Net, self).__init__()
        # 1 input image channel, 6 output channels, 5x5 square convolution
        # kernel
        self.fc1 = nn.Linear(1024, hidden_layer)  # 5*5 from image dimension
        self.fc2 = nn.Linear(hidden_layer, n_class)
        # self.fc3 = nn.Linear(84, 10)

    def forward(self, input):
        # Convolution layer C1: 1 input image channel, 6 output channels,
        # 5x5 square convolution, it uses RELU activation function, and
        # outputs a Tensor with size (N, 6, 28, 28), where N is the size of the batch
        # c1 = F.relu(self.conv1(input))
        # # Subsampling layer S2: 2x2 grid, purely functional,
        # # this layer does not have any parameter, and outputs a (N, 16, 14, 14) Tensor
        # s2 = F.max_pool2d(c1, (2, 2))
        # # Convolution layer C3: 6 input channels, 16 output channels,
        # # 5x5 square convolution, it uses RELU activation function, and
        # # outputs a (N, 16, 10, 10) Tensor
        # c3 = F.relu(self.conv2(s2))
        # # Subsampling layer S4: 2x2 grid, purely functional,
        # # this layer does not have any parameter, and outputs a (N, 16, 5, 5) Tensor
        # s4 = F.max_pool2d(c3, 2)
        # # Flatten operation: purely functional, outputs a (N, 400) Tensor
        # s4 = torch.flatten(s4, 1)
        # # Fully connected layer F5: (N, 400) Tensor input,
        # and outputs a (N, 120) Tensor, it uses RELU activation function
        f5 = F.relu(self.fc1(input))
        # Fully connected layer F6: (N, 120) Tensor input,
        # and outputs a (N, 84) Tensor, it uses RELU activation function
        # f6 = F.relu(self.fc2(f5))
        # Gaussian layer OUTPUT: (N, 84) Tensor input, and
        # outputs a (N, 10) Tensor
        output = self.fc2(f5)
        return output


net = Net()
print(net)

In [ ]:
torch.save(net, 'net.pt')

In [ ]:



class LocalUpdate(object):
    def __init__(self, args=None, dataset=None):
        self.args = args
        self.loss_func = nn.CrossEntropyLoss()
        self.dataset = dataset
        # self.ldr_train = dataloaders['train']
        self.ldr_train = torch.utils.data.DataLoader(dataset=dataset, shuffle=False, batch_size=1)
        self.transform_train = transforms.Compose([transforms.RandomCrop(32, padding=4),transforms.RandomHorizontalFlip(),])

    def train_and_compute_fisher(self, net,backbone, n_c):
        net.train()
        optimizer = torch.optim.SGD(net.parameters(), lr=0.001, momentum = 0.9)
        step_count = 0
        backbone.to(device)
        net.to(device)
        dataset_local=[]
        for epoch in range(20):
            batch_loss = []
            for batch_idx, (images, labels) in enumerate(self.ldr_train):
                images, labels = images.to(device), labels.to(device)
                # if(self.args['augmentation']==True):
                #     images = self.transform_train(images)
                optimizer.zero_grad()
                feats=backbone(images)
                log_probs = net(feats)
                loss = self.loss_func(log_probs, labels)
                loss.backward()
                optimizer.step()
                batch_loss.append(loss.item())
                dataset_local.append(tuple([torch.tensor(feats).squeeze(),labels]))

                # print(f"length of local dataset is {len(dataset_local)}")
                # print(f"shape of local dataset[0][0] is {dataset_local[0][0].shape}")
                # print(f"dataset_local[0][0] is {dataset_local[0][0]}")

            print ("Epoch No. ", epoch, "Loss " , sum(batch_loss)/len(batch_loss))
            dataloader_local=torch.utils.data.DataLoader(dataset=dataset_local, shuffle=False, batch_size=32)
            
        F_kfac = FIM(model=net,
                          loader=dataloader_local,
                          representation=PMatKFAC,
                          device='cuda',
                          n_output=n_c)
        
        F_diag = FIM(model=net,
                          loader=dataloader_local,
                          representation=PMatDiag,
                          device='cuda',
                          n_output=n_c)

        F_diag = F_diag.get_diag()
        vec_curr = parameters_to_vector(net.parameters())            
        return vec_curr, net, F_kfac, F_diag

In [ ]:
local = LocalUpdate(args=None, dataset=image_datasets['train'])
model_vector, model, F_kfac, F_diag = local.train_and_compute_fisher(net,modified_model, num_classes)

In [ ]:
dataset_list=[unlabled_subset,labled_subset]
model_vector_list=[]
model_list=[]
F_kfac_list=[]
F_diag_list=[]
p=0.5

for ds in dataset_list:
    local = LocalUpdate(args=None, dataset=ds)
    net=Net()
    model_vector, model, F_kfac, F_diag = local.train_and_compute_fisher(net,modified_model, n_class)
    
    F_diag = F_diag*p
    for layer_id in F_kfac.data.keys():
        F_kfac.data[layer_id] = list(F_kfac.data[layer_id])

        F_kfac.data[layer_id][0].mul_(p)
        F_kfac.data[layer_id][1].mul_(p)


    model_vector_list.append(model_vector)
    model_list.append(model)
    F_kfac_list.append(F_kfac)
    F_diag_list.append(F_diag)



In [ ]:
F_kfac_list[0]

In [ ]:
args={
"bs":64,
"local_epochs":2,
"device":'cuda',
"rounds":2, 
"num_clients": 2,
"augmentation": False,
"eta": 0.01,
"dataset":dataset_list,
"model":model_name,
"use_pretrained":None,
"n_c":n_class
}


In [ ]:
#generate val loader with val data features
backbone=modified_model
optimizer = torch.optim.SGD(net.parameters(), lr=0.001, momentum = 0.9)
step_count = 0
backbone.to(device)
dataset_feats_val=[]

batch_loss = []
for batch_idx, (images, labels) in enumerate(dataloaders['val']):
    images, labels = images.to(device), labels.to(device)
    # if(self.args['augmentation']==True):
    #     images = self.transform_train(images)
    optimizer.zero_grad()
    feats=backbone(images)
    dataset_feats_val.append(tuple([torch.tensor(feats).squeeze(),labels.item()]))
dataloader_feats_val=torch.utils.data.DataLoader(dataset=dataset_feats_val, shuffle=False, batch_size=32)

In [ ]:
# from FedFisher import get_one_shot_model,get_fisher_merge_model
from FedFisher.run_one_shot_algs import get_fisher_diag_model
net_global=Net()
ptv= parameters_to_vector(net_global.parameters()).numel()

cun_cli=2
p=[0.5,0.5]
net_fed_fisher=get_fisher_diag_model(ptv, cun_cli, p, args, net_global, model_vector_list, F_diag_list,dataset_feats_val)

In [ ]:
import FedFisher

In [ ]:
from FedFisher.run_one_shot_algs import get_fedavg_model
net_global_fed_avg=Net()
net_global_fed_avg=get_fedavg_model(ptv, cun_cli, p, args, net_global_fed_avg, model_vector_list)

In [ ]:
net_global.fc1.weight.shape,net_global.fc1.bias.shape,net_global.fc2.weight.shape,net_global.fc2.bias.shape


In [ ]:
def test_img(net_g, data_loader, args):
    net_g.eval()
    test_loss = 0
    correct = 0
    
    # data_loader = DataLoader(datatest, batch_size=32)
    l = len(data_loader)
    for idx, (data, target) in enumerate(data_loader):
        data, target = data.to(device), target.to(device)
        log_probs = net_g(data)
        test_loss += F.cross_entropy(log_probs, target, reduction = 'sum').item()
        y_pred = log_probs.data.max(1, keepdim=True)[1]
        correct += y_pred.eq(target.data.view_as(y_pred)).long().cpu().sum()

        
    test_loss /= len(data_loader.dataset)
    accuracy = 100.00 * correct / len(data_loader.dataset)
    return accuracy.numpy(), test_loss


In [ ]:
test_img(net_global_fed_avg, dataloader_feats_val, args),test_img(net_fed_fisher, dataloader_feats_val, args)


# dataloader_feats_val

In [ ]:
import pickle

with open("net_global.pkl", "wb") as f:
    pickle.dump(net_fed_fisher.state_dict(), f)

In [ ]:
#test function for fed model
from sklearn.metrics import precision_recall_fscore_support
def predict_model_op_fed(ds,clf):
    true_lables= [lbl for (img,lbl) in ds]
    dense_op=modified_model(ds)
    y_true= true_lables
    y_pred= clf.predict(dense_op)
    print("precision,recall,accuracy,f1 score:")
    print(precision_recall_fscore_support(y_true, y_pred, average='macro'))
    # return clf.predict_proba(dense_op)

In [ ]:
net_global.fc1.weight.detach().numpy().shape

In [ ]:
model_vector, model, F_kfac, F_diag

FEDFISHER END

MULTI CLIENT START

In [ ]:
import socket

client_list=[]
num_clients=2
host="10.107.47.157"
port=5002
BUFFER_SIZE=4096
s=socket.socket()
s.bind((host,port))

s.listen(num_clients)

while(len(client_list)<num_clients):
    client_socket, client_address = s.accept()
    print(f"Connection established with {client_address}")
    client_list.append(client_socket)



In [ ]:
import subprocess
msg="sending_c"
for cli in client_list:
    cli.send(msg.encode())
#scp c (pt)
command = "scp net.pt mihir1@10.107.47.132:/home/mihir1/Downloads/abhinavModels/"
output = subprocess.run(command, shell=True, check=True)

command = "scp net.pt mihir1@10.107.47.182:/home/mihir1/Downloads/abhinavModels/"
output = subprocess.run(command, shell=True, check=True)

In [ ]:
msg="run_inf"
for cli in client_list:
    cli.send(msg.encode())


In [ ]:
msg="infer_train_test"
for cli in client_list:
    cli.send(msg.encode())


In [ ]:


msg="infer_only_test"
for cli in client_list:
    cli.send(msg.encode())

In [ ]:
msg="sending_b"
for cli in client_list:
    cli.send(msg.encode())
# client_socket.send(msg.encode())
#scp b (onnx)

command = "scp model123.onnx mihir1@10.107.47.132:/home/mihir1/Downloads/abhinavModels/"
output = subprocess.run(command, shell=True, check=True)

command = "scp model123.onnx mihir1@10.107.47.182:/home/mihir1/Downloads/abhinavModels/"
output = subprocess.run(command, shell=True, check=True)


In [ ]:
msg="sending_c"
for cli in client_list:
    cli.send(msg.encode())
# client_socket.send(msg.encode())
#scp c (pt)
command = "scp net.pt mihir@10.107.47.175:/home/mihir/Downloads/abhinavModels/"
output = subprocess.run(command, shell=True, check=True)


In [ ]:
msg="infer_only_test"
for cli in client_list:
    cli.send(msg.encode())

In [ ]:
s.close()
for cli in client_list:
    cli.close()

MULTI CLIENT END

FEDERATE MULTICLIENT

SENDING 'FED_RQST' COMMAND WILL GET THE MODELS FROM BOTH CLIENT. (NET1.PT AND NET2.PT). AFTER THAT THE AGGREGATION IS PERFORMED

In [ ]:
msg="fed_rqst"
for cli in client_list:

    cli.send(msg.encode())

In [ ]:
dataset_list=[unlabled_subset,labled_subset]
args={
"bs":64,
"local_epochs":2,
"device":'cuda',
"rounds":2, 
"num_clients": 2,
"augmentation": False,
"eta": 0.01,
"dataset":dataset_list,
"model":model_name,
"use_pretrained":None,
"n_c":n_class
}


In [ ]:
#compute fisher

import torch
import pickle
n_c=n_class


model_vector_list=[]
model_list=[]
F_kfac_list=[]
F_diag_list=[]



dataset_local_list=[]
for i in range(0,2):
    idx=i+1
    with open('feat_dataset_'+str(idx)+'.pkl', 'rb') as f:
        dataset_local = pickle.load(f)
        dataset_local_list.append(dataset_local)

net_list= [torch.load('net1.pt').to(device),torch.load('net2.pt').to(device)]


tot=len(dataset_local_list[0])+len(dataset_local_list[1])
p=0.5

dataloader_local_list=[torch.utils.data.DataLoader(dataset=ds, shuffle=False, batch_size=32) for ds in dataset_local_list]

for i in enumerate (net_list):
    idx=i[0]

    mdl=net_list[idx]

    F_kfac = FIM(model=mdl,
                            loader=dataloader_local_list[idx],
                            representation=PMatKFAC,
                            device='cuda',
                            n_output=n_c)
            
    F_diag = FIM(model=mdl,
                            loader=dataloader_local_list[idx],
                            representation=PMatDiag,
                            device='cuda',
                            n_output=n_c)

    F_diag = F_diag.get_diag()
    vec_curr = parameters_to_vector(mdl.parameters()) 

    F_diag = F_diag*p
    for layer_id in F_kfac.data.keys():
        F_kfac.data[layer_id] = list(F_kfac.data[layer_id])

        F_kfac.data[layer_id][0].mul_(p)
        F_kfac.data[layer_id][1].mul_(p)


    model_vector_list.append(vec_curr)
    model_list.append(mdl)
    F_kfac_list.append(F_kfac)
    F_diag_list.append(F_diag)



In [ ]:
p

In [ ]:
#generate val loader with val data features
backbone=modified_model
optimizer = torch.optim.SGD(net.parameters(), lr=0.001, momentum = 0.9)
step_count = 0
backbone.to(device)
dataset_feats_val=[]

batch_loss = []
for batch_idx, (images, labels) in enumerate(dataloaders['val']):
    images, labels = images.to(device), labels.to(device)
    # if(self.args['augmentation']==True):
    #     images = self.transform_train(images)
    optimizer.zero_grad()
    feats=backbone(images)
    dataset_feats_val.append(tuple([torch.tensor(feats).squeeze(),labels.item()]))
dataloader_feats_val=torch.utils.data.DataLoader(dataset=dataset_feats_val, shuffle=False, batch_size=32)

In [ ]:
F_diag.shape

In [ ]:
len(dataset_local_list[0])

In [ ]:
# from FedFisher import get_one_shot_model,get_fisher_merge_model
from FedFisher.run_one_shot_algs import get_fisher_diag_model
net_global=Net()
ptv= parameters_to_vector(net_global.parameters()).numel()

cun_cli=2
p=[0.5,0.5]
net_fed_fisher=get_fisher_diag_model(ptv, cun_cli, p, args, net_global, model_vector_list, F_diag_list,dataset_feats_val)

In [ ]:
from FedFisher.run_one_shot_algs import get_fedavg_model
net_global_fed_avg=Net()
net_global_fed_avg=get_fedavg_model(ptv, cun_cli, p, args, net_global_fed_avg, model_vector_list)

In [ ]:
def test_img(net_g, data_loader, args):
    net_g.eval()
    test_loss = 0
    correct = 0
    
    # data_loader = DataLoader(datatest, batch_size=32)
    l = len(data_loader)
    for idx, (data, target) in enumerate(data_loader):
        data, target = data.to(device), target.to(device)
        log_probs = net_g(data)
        test_loss += F.cross_entropy(log_probs, target, reduction = 'sum').item()
        y_pred = log_probs.data.max(1, keepdim=True)[1]
        correct += y_pred.eq(target.data.view_as(y_pred)).long().cpu().sum()

        
    test_loss /= len(data_loader.dataset)
    accuracy = 100.00 * correct / len(data_loader.dataset)
    return accuracy.numpy(), test_loss


In [ ]:
test_img(net_global_fed_avg, dataloader_feats_val, args),test_img(net_fed_fisher, dataloader_feats_val, args)

In [ ]:
torch.save(net_global_fed_avg,'net_global_fed_avg.pt'),torch.save(net_fed_fisher,'net_fed_fisher.pt'),

In [ ]:
from time import sleep

In [ ]:
import subprocess
from time import sleep
rqst="sending_global"
for cli in client_list:
        cli.send(rqst.encode())
        #do scp

stratergy='fedavg'

if(stratergy=='fedavg'):
        fedmodelname='net_global_fed_avg'
if(stratergy=='fedfisher'):
        fedmodelname='net_fed_fisher'


command = "scp "+str(fedmodelname)+".pt mihir1@10.107.47.132:/home/mihir1/Downloads/abhinavModels/"
output = subprocess.run(command, shell=True, check=True)
sleep(5)
command = "scp "+str(fedmodelname)+".pt mihir1@10.107.47.182:/home/mihir1/Downloads/abhinavModels/"
output = subprocess.run(command, shell=True, check=True)
sleep(5)

rqst="model_sent"
for cli in client_list:
        cli.send(rqst.encode())

In [ ]:
msg="infer_train_test"
for cli in client_list:

    cli.send(msg.encode())

    

In [ ]:
msg="fed_rqst"
for cli in client_list:

    cli.send(msg.encode())

In [ ]:
s.close()
for cli in client_list:
    cli.close()

END FEDERATE MULTI CLIENT

In [ ]:
# selected_for_training_indices=selected_for_training_indices[0]
